# Data Collection: Impacts of Data Centers on Electrical Utilities

This notebook downloads the raw data for our project and saves it to the `data/raw/` folder.

| Dataset | What it gives us | Saved as |
|---|---|---|
| FracTracker U.S. Data Centers | Location, status, size, MW and operator for each data center | `fractracker_data_centers.csv` |
| FracTracker Crypto Mining Operations | Known crypto mining sites (labels for Part A) | `fractracker_crypto.csv` |
| EIA-860 Plant and Generator Data | Power plants: location, operating year, energy source | `eia860_2025/` |
| EIA-930 Demand and Generation Data | Hourly demand and generation by grid region and fuel type | `eia930/` |
| EIA-861M Retail Sales of Electricity | Monthly electricity sold by state and sector | `eia861m_sales_revenue.xlsx` |

In [3]:
import requests
import zipfile
import pandas as pd
from pathlib import Path

In [4]:
DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Collect data

Links for each data source. The files in `file_urls` can be downloaded directly. The data center and EIA-930 links need extra steps, which are handled in their own sections below.

In [5]:
file_urls = {
  "fractracker_crypto.csv": "https://docs.google.com/spreadsheets/d/e/2PACX-1vRIG7TvcdFYbBnqUl2--Y7qPwl-9AixBXPPKeMrnr0X_1SnPLGxhJx6PAZZ5Ppi02u_4vNMsdazES3p/pub?output=csv",
  "eia860_2025.zip": "https://www.eia.gov/electricity/data/eia860/xls/eia8602025.zip",
  "eia861m_sales_revenue.xlsx": "https://www.eia.gov/electricity/data/eia861m/xls/sales_revenue.xlsx"
}

data_centers_url = "https://services.arcgis.com/jDGuO8tYggdCCnUJ/arcgis/rest/services/data_centers_v4_agol_all/FeatureServer/0/query"
demand_url = "https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/EIA930_{kind}_{year}_{half}.csv"

In [6]:
for filename, url in file_urls.items():
    response = requests.get(url)
    (DATA_DIR / filename).write_bytes(response.content)
    print("Saved:", filename)

Saved: fractracker_crypto.csv
Saved: eia860_2025.zip
Saved: eia861m_sales_revenue.xlsx


### EIA-860 Plant and Generator Data

Unzip the download so we can read the Excel files inside.

In [7]:
zip_file = zipfile.ZipFile(DATA_DIR / "eia860_2025.zip")
zip_file.extractall(DATA_DIR / "eia860_2025")
zip_file.close()
print("Unzipped: eia860_2025.zip")

Unzipped: eia860_2025.zip


### FracTracker U.S. Data Centers

The website only gives us 1,000 data centers at a time, so we keep asking for the next 1,000 until there are none left.

In [8]:
all_rows = []
start = 0

while True:
    params = {
        "where": "1=1",
        "outFields": "*",
        "f": "json",
        "resultOffset": start
    }
    response = requests.get(data_centers_url, params=params)
    features = response.json()["features"]

    # stop when there are no more data centers
    if len(features) == 0:
        break

    for feature in features:
        all_rows.append(feature["attributes"])

    start = start + len(features)

data_centers = pd.DataFrame(all_rows)
data_centers.to_csv(DATA_DIR / "fractracker_data_centers.csv", index=False)
print("Saved: fractracker_data_centers.csv with", len(data_centers), "rows")

Saved: fractracker_data_centers.csv with 1719 rows


### EIA-930 Demand and Generation Data

EIA splits this data into one file per half year.
- `BALANCE` files have demand, forecast, interchange and generation by fuel type (starts July 2015).
- `SUBREGION` files have demand for the Texas (ERCOT) regions (starts July 2018).

If a file does not exist yet, EIA sends back a web page instead of a CSV, so we skip it.

In [9]:
EIA930_DIR = DATA_DIR / "eia930"
EIA930_DIR.mkdir(exist_ok=True)

kinds = ["BALANCE", "SUBREGION"]
years = range(2015, 2027)
halves = ["Jan_Jun", "Jul_Dec"]

for kind in kinds:
    for year in years:
        for half in halves:
            filename = f"EIA930_{kind}_{year}_{half}.csv"
            url = demand_url.format(kind=kind, year=year, half=half)
            response = requests.get(url)

            if "html" in response.headers["Content-Type"]:
                print("Not available:", filename)
            else:
                (EIA930_DIR / filename).write_bytes(response.content)
                print("Saved:", filename)

Not available: EIA930_BALANCE_2015_Jan_Jun.csv
Saved: EIA930_BALANCE_2015_Jul_Dec.csv
Saved: EIA930_BALANCE_2016_Jan_Jun.csv
Saved: EIA930_BALANCE_2016_Jul_Dec.csv
Saved: EIA930_BALANCE_2017_Jan_Jun.csv
Saved: EIA930_BALANCE_2017_Jul_Dec.csv
Saved: EIA930_BALANCE_2018_Jan_Jun.csv
Saved: EIA930_BALANCE_2018_Jul_Dec.csv
Saved: EIA930_BALANCE_2019_Jan_Jun.csv
Saved: EIA930_BALANCE_2019_Jul_Dec.csv
Saved: EIA930_BALANCE_2020_Jan_Jun.csv
Saved: EIA930_BALANCE_2020_Jul_Dec.csv
Saved: EIA930_BALANCE_2021_Jan_Jun.csv
Saved: EIA930_BALANCE_2021_Jul_Dec.csv
Saved: EIA930_BALANCE_2022_Jan_Jun.csv
Saved: EIA930_BALANCE_2022_Jul_Dec.csv
Saved: EIA930_BALANCE_2023_Jan_Jun.csv
Saved: EIA930_BALANCE_2023_Jul_Dec.csv
Saved: EIA930_BALANCE_2024_Jan_Jun.csv
Saved: EIA930_BALANCE_2024_Jul_Dec.csv
Saved: EIA930_BALANCE_2025_Jan_Jun.csv
Saved: EIA930_BALANCE_2025_Jul_Dec.csv
Saved: EIA930_BALANCE_2026_Jan_Jun.csv
Saved: EIA930_BALANCE_2026_Jul_Dec.csv
Not available: EIA930_SUBREGION_2015_Jan_Jun.csv
Not ava

### Compute Atlas Facilities

This file has to be downloaded by hand.

1. Go to https://www.compute-atlas.com/data.
2. Click CSV
3. Move the downloaded `compute-atlas-facilities.csv` into the `data/raw/` folder.

## Check the downloads

Prints the number of rows in each downloaded file next to the number we expect, so we can tell if everyone has the same data.

In [10]:
expected_rows = {
    "fractracker_data_centers.csv": 1708,
    "fractracker_crypto.csv": 302,
    "compute-atlas-facilities.csv": 2659
}

for filename, expected in expected_rows.items():
    rows = len(pd.read_csv(DATA_DIR / filename))
    print(f"{filename}: {rows} rows (expected {expected})")

fractracker_data_centers.csv: 1719 rows (expected 1708)
fractracker_crypto.csv: 302 rows (expected 302)
compute-atlas-facilities.csv: 2659 rows (expected 2659)
